# NUTDTS 816 Time Series Analysis
## L14 Combinations, hierarchies, competitions

Lab notebook for Chapter 7 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### Carried forward from Lab 13 (run these cells first; they define the objects used below)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import tsdata
def accuracy(actual, forecast, train, m=1):
    e = actual.values - forecast.values
    q = np.mean(np.abs(train.values[m:] - train.values[:-m]))
    out = {'MAE': np.mean(np.abs(e)), 'RMSE': np.sqrt(np.mean(e**2)), 'MASE': np.mean(np.abs(e)) / q}
    if (actual.values > 0).all(): out['MAPE'] = 100 * np.mean(np.abs(e / actual.values))
    return pd.Series(out)

# Illustration: MAPE rewards under-forecasting. Next month's value will be 100 or 200 with equal probability (mean 150).
for f_ in [100, 125, 150, 175, 200]:
    print(f'forecast {f_}: expected MAPE = {50 * (abs(100 - f_) / 100 + abs(200 - f_) / 200):5.1f}%   expected MAE = {0.5 * (abs(100 - f_) + abs(200 - f_)):5.1f}')

In [ ]:
def rolling_origin(y, fit_forecast, h, first_origin, step=1, window=None, m=1):
    """Generic rolling-origin evaluation.
    fit_forecast(train_series, h) -> np.array of h forecasts. Returns a DataFrame of errors, rows = origins, cols = horizons 1..h."""
    errs, origins = [], []
    for T in range(first_origin, len(y) - h + 1, step):
        train = y.iloc[max(0, T - window) if window else 0:T]
        fc = np.asarray(fit_forecast(train, h))
        errs.append(y.iloc[T:T + h].values - fc); origins.append(y.index[T - 1])
    E = pd.DataFrame(errs, index=origins, columns=[f'h={k}' for k in range(1, h + 1)])
    q = np.mean(np.abs(y.values[m:] - y.values[:-m]))    # scaling for MASE, from the whole series
    return E, q

def summarise(E, q):
    return pd.DataFrame({'MAE': E.abs().mean(), 'RMSE': np.sqrt((E**2).mean()), 'MASE': E.abs().mean() / q})

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
cpi = tsdata.nigeria_cpi(); infl = (100 * np.log(cpi).diff()).dropna(); infl.name = 'inflation'
h, first = 12, 72      # forecast a year ahead from origins starting after six years of data
methods = {
    'Naive':      lambda tr, h: np.repeat(tr.iloc[-1], h),
    'Mean (24m)': lambda tr, h: np.repeat(tr.iloc[-24:].mean(), h),
    'ETS(A,N,N)': lambda tr, h: ETSModel(tr, error='add', initialization_method='estimated').fit(disp=False).forecast(h).values,
    'ARIMA(1,0,1)': lambda tr, h: ARIMA(tr, order=(1, 0, 1)).fit().forecast(h).values,
    'ARIMA(0,1,1)': lambda tr, h: ARIMA(tr, order=(0, 1, 1)).fit().forecast(h).values,
}
res = {}
for name, f in methods.items():
    E, q = rolling_origin(infl, f, h, first_origin=first, step=2); res[name] = (E, q)
mase = pd.DataFrame({name: summarise(E, q)['MASE'] for name, (E, q) in res.items()})
print(f'{len(res["Naive"][0])} forecast origins, expanding window.  MASE by horizon (lower is better; 1 = naive in-sample):')
print(mase.round(3).to_string())
print('\nAverage MASE over horizons 1-12:'); print(mase.mean().round(3).sort_values().to_string())

In [ ]:
ax = mase.plot(figsize=(8.5, 3.4), marker='o', ms=3, lw=1.5)
ax.set_xlabel('forecast horizon (months)'); ax.set_ylabel('MASE'); ax.set_title('Rolling-origin accuracy by horizon: monthly inflation (simulated)'); ax.legend(fontsize=8)
_caption = 'Accuracy by horizon separates methods that a single hold-out would conflate. Everything degrades with horizon; the differences between methods are largest at short horizons.'

In [ ]:
def interval_eval(y, fit_pi, h, first_origin, step, alpha=0.2):
    """fit_pi(train, h) -> (mean, lower, upper) arrays for a (1-alpha) interval. Returns coverage and mean Winkler score by horizon."""
    cov, wink = [], []
    for T in range(first_origin, len(y) - h + 1, step):
        tr = y.iloc[:T]; act = y.iloc[T:T + h].values
        mean_, lo, hi = fit_pi(tr, h)
        inside = (act >= lo) & (act <= hi)
        w = (hi - lo) + (2 / alpha) * (lo - act) * (act < lo) + (2 / alpha) * (act - hi) * (act > hi)
        cov.append(inside); wink.append(w)
    return pd.DataFrame({'coverage': np.mean(cov, axis=0), 'Winkler': np.mean(wink, axis=0)}, index=[f'h={k}' for k in range(1, h + 1)])

def arima_pi(tr, h, order=(1, 0, 1), alpha=0.2):
    f = ARIMA(tr, order=order).fit().get_forecast(h); ci = f.conf_int(alpha=alpha)
    return f.predicted_mean.values, ci.iloc[:, 0].values, ci.iloc[:, 1].values
def naive_pi(tr, h, alpha=0.2):           # empirical interval for the naive method from the distribution of past h-step changes
    mean_ = np.repeat(tr.iloc[-1], h); lo, hi = [], []
    for k in range(1, h + 1):
        ch = (tr - tr.shift(k)).dropna(); lo.append(mean_[k-1] + ch.quantile(alpha / 2)); hi.append(mean_[k-1] + ch.quantile(1 - alpha / 2))
    return mean_, np.array(lo), np.array(hi)

ev = pd.concat({'ARIMA(1,0,1) 80% PI': interval_eval(infl, arima_pi, h, first, 2), 'Naive, empirical 80% PI': interval_eval(infl, naive_pi, h, first, 2)}, axis=1)
print(ev.round(2).iloc[[0, 2, 5, 8, 11]].to_string())

In [ ]:
from scipy import stats
def diebold_mariano(e1, e2, h=1, loss=np.abs):
    d = loss(e1) - loss(e2); n = len(d); dbar = d.mean()
    # long-run variance with Bartlett weights up to lag h-1 (Diebold-Mariano 1995)
    gamma = [np.sum((d[k:] - dbar) * (d[:n - k] - dbar)) / n for k in range(h)]
    lrv = gamma[0] + 2 * sum((1 - k / h) * gamma[k] for k in range(1, h))
    dm = dbar / np.sqrt(lrv / n); return dm, 2 * (1 - stats.norm.cdf(abs(dm)))
for hh in [1, 6, 12]:
    e_naive = res['Naive'][0][f'h={hh}'].values; e_arima = res['ARIMA(1,0,1)'][0][f'h={hh}'].values
    dm, p = diebold_mariano(e_naive, e_arima, h=hh)
    print(f'h = {hh:2d}: DM statistic = {dm:5.2f}, p = {p:.3f}  (positive = ARIMA(1,0,1) more accurate than naive)')

## Combinations, hierarchies, competitions

### 7.6 Forecast combinations

In [ ]:
E_ets, E_ar, E_nv = res['ETS(A,N,N)'][0], res['ARIMA(1,0,1)'][0], res['Naive'][0]
# Errors of the equal-weight combination = average of the errors (same actuals)
E_comb = (E_ets + E_ar + E_nv) / 3; E_comb2 = (E_ets + E_ar) / 2
q = res['Naive'][1]
tab = pd.DataFrame({'Naive': summarise(E_nv, q)['MASE'], 'ETS': summarise(E_ets, q)['MASE'], 'ARIMA(1,0,1)': summarise(E_ar, q)['MASE'],
                    'Mean of ETS+ARIMA': summarise(E_comb2, q)['MASE'], 'Mean of all three': summarise(E_comb, q)['MASE']})
print('MASE by horizon:'); print(tab.round(3).iloc[[0, 2, 5, 11]].to_string()); print('\nAverage over horizons:'); print(tab.mean().round(3).to_string())

## Exercises

5. Implement the Diebold-Mariano test for squared-error loss and apply it to the ETS-versus-combination comparison at horizons 1 and 6.
6. Explain to a regional manager why the sum of the best forecasts for each of their 12 depots is not the best forecast for the region, and what reconciliation does about it.
7. The M4 winner combined exponential smoothing with a recurrent network trained across all 100,000 series. Explain, in terms of what each component contributes, why this hybrid beat both pure ETS and pure RNN approaches.

In [ ]:
# Your work here
